# LoopMoE 1B (Test Branch) — Colab TPU v5e-1 Benchmark & Profiler

Tests the 1B 1-Layer DeepSeek-V4.1-Flash LoopMoE model uniformly at **4K sequence length (`R=1`)** with **40% EN / 40% ZH / 10% Code / 10% Math** (UltraData L3) on **TPU v5e-1** (`16 GB HBM`, `197 TFLOPS BF16`, `819 GB/s`, `128x128 MXU`) with:
- Memory-safe & fast v5e-1 preset (`microbatch=8` = 32,768 tok, `grad_accum=8` = 262,144 global tok, `chunk_ce_size=8192`, `remat_moe=True`, `remat_sublayers=False`)
- XLA `memory_analysis` + `cost_analysis` (arithmetic intensity & compute/memory bound diagnosis)
- Per-module latency breakdown (`engram`, `mhc`, `iter_adaln`, `mla`, `moe_sort`, `moe_total`, `chunked_ce_loss`, `muon_adamw`)
- **JAX / XProf** timeline trace capture (`tensorboard` & **Perfetto** `ui.perfetto.dev`)

In [ ]:
# 1. Clone/update Test branch and install latest pinned stack from requirements.txt
import os
REPO_DIR = '/content/M3_Test'
if not os.path.exists(REPO_DIR):
    !git clone -b Test https://github.com/GarD0m/M3.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull origin Test

!pip install -q -U -r {REPO_DIR}/requirements.txt

In [ ]:
# 2. Probe TPU in a subprocess so the Jupyter kernel does NOT lock libtpu.so
!python -c "import jax; print('Backend:', jax.default_backend(), '| Devices:', jax.devices()); [print('Memory:', d.memory_stats()) for d in jax.local_devices()]"

In [ ]:
# 3. Run TPU v5e-1 MFU benchmark + per-module breakdown + XProf profiler trace
%cd {REPO_DIR}
!python scripts/bench_tpu_mfu.py \
    --tpu-preset v5e-1 \
    --seq-lens 4096 \
    --loops 1 \
    --microbatch 8 \
    --grad-accum 8 \
    --chunk-ce-size 8192 \
    --remat-moe \
    --no-remat-sublayers \
    --warmup-steps 2 \
    --bench-steps 5 \
    --breakdown \
    --breakdown-seq-len 4096 \
    --profile-dir tpu_profiles/v5e_4k_mb8 \
    --profile-seq-len 4096 \
    --profile-steps 3 \
    --out-json bench_v5e_report.json

In [ ]:
# 4. Optional: stress-test microbatch=16 (65,536 tok) with grad_accum=4 on v5e-1
%cd {REPO_DIR}
!python scripts/bench_tpu_mfu.py \
    --tpu-preset v5e-1 \
    --seq-lens 4096 \
    --loops 1 \
    --microbatch 16 \
    --chunk-ce-size 4096 \
    --remat-sublayers \
    --grad-accum 4 \
    --warmup-steps 2 \
    --bench-steps 5 \
    --breakdown \
    --out-json bench_v5e_mb16.json

In [ ]:
# 5. Optional: 4K real UltraData L3 streaming test (40% EN / 40% ZH / 10% Code / 10% Math)
# Add --english-only for 80% EN / 0% ZH / 10% Code / 10% Math (uses artifacts/tokenizer_64k_en/tokenizer.json)
%cd {REPO_DIR}
!python scripts/train_tpu.py \
    --tpu-preset v5e-1 \
    --data-mode stream \
    --start-stage 1 --end-stage 1 \
    --max-steps-per-stage 20 \
    --log-every 5 \
    --profile-dir tpu_profiles/v5e_4k_stream

In [ ]:
# 6. View Profiler in TensorBoard XProf or download trace for Perfetto (https://ui.perfetto.dev)
%load_ext tensorboard
%tensorboard --logdir {REPO_DIR}/tpu_profiles/v5e_4k

# To download trace archive for ui.perfetto.dev:
# !zip -r /content/v5e_profile_trace.zip {REPO_DIR}/tpu_profiles/v5e_4k {REPO_DIR}/bench_v5e_report.json